<a href="https://colab.research.google.com/github/Arnak101/Arnak101.github.io/blob/main/Hometasks/Pro/AI_HW6_uplift.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1> Задание по Uplift-моделированию </h1>

<h2>Введение</h2>

Перед вами типичная задача, возникающая при работе с моделями кампейнинга в банке: заказчик запустил несколько пилотов по взаимодействию с клиентами с помощью разных каналов: push в мобильном приложении, sms, баннеры в мобильном приложении и реклама в других приложениях экосистемы. Заказчик хотел бы понимать, какой канал взаимодействия с клиентом наиболее эффективен для каждого клиента из клиентской базы. Кампании планируются и запускаются в ежемесячном режиме. Иными словами, заказчик хотел бы в идеале ежемесячно получать список клиентов, которым необходимо отправить коммуникацию с указанием канала и прироста вероятности покупки в случае, если клиенту отправят коммуникацию по сравнению с тем случаем, когда клиенту коммуникацию не отправят.

<b>Таким образом: </b>
1.	У нас есть база клиентов (клиенты, имеющие id в банке). По данной базе осуществляется рассылка тех или иных стимулирующих коммуникаций по различным продуктам, каналам (например SMS, Push, баннеры в мобильном приложении и т.д.) и сегментам клиентов
2.	Признаковое описание клиента состоит из различных агрегатов действий клиента за месяц или его объективных характеристик: например, средняя сумма средств на депозитах за месяц, среднее число кликов клиента в день за месяц в разделе "инвестиции" в мобильном приложении или возраст клиента
3.	При формировании обучающей/тестовой выборки допускается, что один и тот же клиент за разные месяцы — это разные объекты. То есть допускается, что клиент в феврале и клиент в марте — это разные клиенты (то есть мы можем оперировать с ними как с разными сущностями).
4.	Агрегаты действий клиента за месяц появляются примерно 10 числа следующего месяца. То есть, например, агрегаты за декабрь появляются 10 января. В свою очередь списки клиентов, которым необходимо осуществить рассылку должны быть сформированы ориентировочно 20 числа предыдущего месяца. Таким образом, <b> модель должна быть обучена делать предсказания с лагом в два месяца </b>, то есть должна делать предсказание на март по клиентским агрегатам за январь. Обязательно учтите это при обучении модели (в противном случае можно получить лик таргета, так как часто величину, которую мы предсказываем уже есть в клиентских агрегатах, но смещенная на два месяца).


## Оценивание задания:

Всего за задание можно получить 50 первичных баллов, которые затем переводятся в 10-балльную шкалу делением не 5.

Скачаем архив с данными по ссылке и разархивируем.

In [1]:
!pip install gdown -q

In [2]:
import gdown

url = 'https://drive.google.com/uc?id=19nKGaxm3RwHxh2UWPo537_-MDx21AkHO'
output = 'Data.zip'
gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=19nKGaxm3RwHxh2UWPo537_-MDx21AkHO
From (redirected): https://drive.google.com/uc?id=19nKGaxm3RwHxh2UWPo537_-MDx21AkHO&confirm=t&uuid=b4e3f110-4e53-4159-9d17-3d8f33843166
To: /content/Data.zip
100%|██████████| 289M/289M [00:03<00:00, 72.2MB/s]


'Data.zip'

In [3]:
import zipfile

with zipfile.ZipFile('Data.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

<h2>Описание данных</h2>

Перед вами несколько наборов данных, на основе которых вам будет необходимо обучить Uplift модели, сделать прогноз на нужный месяц и решить, кому из клиентов отправлять коммуникацию, а кому коммуникацию отправлять не следует.

<h3>Features </h3> Признаки клиентов, клиентские агрегаты, которые описывают поведение клиентов <br>

1. user_id - id клиента
2. report_dt - месяц, на который актуальны признаки
3. city - город, в котором живет клиент
4. age - возраст клиента
5. x1 – x9 - числовые признаки клиента, характеризующие поведение клиента

Первичный ключ таблицы - user_id + report_dt

<h3> Contracts </h3> Таблица с покупками продуктов.

1. contract_id - id покупки
2. user_id - id пользователя, который совершил покупку
3. product_id - id продукта, который был куплен
4. contract_ts – дата момента, когда была совершена покупка

Первичный ключ - contract_id


<h3> Campaings </h3> Кампании, которые проводились (под кампанией мы понимаем рассылку sms, push и т.д).

1. campaing_id - id кампании, первичный ключ таблицы
2. product_id - продукт, по которому проводилась кампания (считаем, что продукты не конкурируют друг с другом)
3. channel - канал, в котором проводилась кампания


<h3> People_in_campaings </h3> Люди, которые принимали участие в кампаниях.

1. campaing_id - id кампании
2. user_id - id пользователя, который попал в кампанию
3. флаг целевой (1) и контрольной (0) группы (целевая группа - это те, кто получил коммуникацию, а контрольная - те, кто нет)
4. delivery_ts - timestamp, когда клиенту фактически была доставлена коммуникация (для контрольной группы nan, подумайте почему)

Первичный ключ данной таблицы - user_id + campaing_id


<h3> Contracts </h3> Таблица с покупками продуктов

1. contract_id - id покупки
2. user_id - id пользователя, который совершил покупку
3. product_id - id продукта, который был куплен
4. contract_ts – дата момента, когда была совершена покупка

Первичный ключ - contract_id


<h1> Постановка задачи </h1> В ноябре 2024 проводилось несколько кампаний по продукту с id 0001 (фактически клиенту рассылалось одно и тоже сообщение, но в разных каналах). Вам необходимо по данным кампаниям построить модель, которая будет определять лучший канал коммуникации каждого клиента и определить, кому из клиентов в марте 2025 отправить какую коммуникацию, а кому коммуникацию вообще отправлять не следует.
Ответ нужно представить в следующем виде (report_dt – дата фичей):

<table>
  <thead>
    <tr>
      <th>user_id</th>
      <th>report_dt</th>
      <th>channel</th>
      <th>uplift</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>10045</td>
      <td>2025-01-31</td>
      <td>banner</td>
      <td>0.07</td>
    </tr>
    <tr>
      <td>10046</td>
      <td>2025-01-31</td>
      <td>no_comm</td>
      <td>0.00</td>
    </tr>
    <tr>
      <td>10047</td>
      <td>2025-01-31</td>
      <td>sms</td>
      <td>0.23</td>
    </tr>
    <tr>
      <td>10048</td>
      <td>2025-01-31</td>
      <td>push</td>
      <td>0.19</td>
    </tr>
  </tbody>
</table>

<h1> Декомпозиция задачи </h1>

<h2> 1.	Сбор и анализ таргета (18 баллов)</h2>

Прежде всего, вам необходимо собрать целевое событие, которое вы собираетесь прогнозировать. В данном случае целевое событие - это покупка продукта 0001 пользователем, участвовавшем в кампании. Обратите внимание, что не все пользователи получают коммуникацию одновременно (delivery_ts в таблице People_in_campaings). Согласно правилу, согласованному с заказчиком, <b> человек из целевой группы купил продукт после коммуникации - это значит, что он купил его в течение 2х недель после получения сообщения, а человек из контрольной - в течение 3х недель с момента старта кампании (старт кампании - начало месяца). </b> То есть для определенной кампании, для каждого клиента, попавшего в кампанию, вам надо будет найти его покупки данного продукта, а потом основываяся на данном правиле превратить покупки в 0 или 1. <br> На выходе у вас должен появиться таблица с целевым действием для каждого канала (колонки client_id, report_dt,  target), где таргет - это бинарная переменная (0 или 1). Колонка report_dt вам нужна как техническая колонка для дальнейших джоинов.<br><br>

Проведите анализ полученных данных (до присоединения клиентских агрегатов). Какие проблемы и сложности в данных вы обнаружили? Что с ними можно сделать? Какая из кампаний наиболее эффективная? Подготовьте выводы по полученным инсайтам.


**Комментарий по заданиям и оцениванию:**

* Вы должны самостоятельно сделать join нескольких таблиц, самостоятельно собрать целевое действие

* Представлены 4 различных канала, за таргет по каждому из каналов можно получить **максимум 2 балла**:
    * 1 балл за то, что просчитано целевое действие для целевой группы (покупка в
течение одной-двух недель с момента получения коммуникации)
    * 1 балл за то, что просчитано целевое действие для контрольной группы (покупка в течение двух-трех недель с момента старта кампании) и сделана таблица в требуемом формате

* Обратите внимание, что не во всех кампаниях содержатся корректные данные для проведения моделирования, и вам необходимо провести анализ данных и в случае выявленных некорректностей - описать их, и не проводить моделирование для "сломанной" кампании  
    * За данный анализ можно получить **8 баллов**

* Вы должны оценить эффективность кампаний по uplift (cреднее значение таргета в целевой минус среднее значение таргета в контрольной группе)
    * За данный анализ можно получить **2 балла**

In [60]:
!pip install scikit-uplift -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.0 MB/s eta 0:00:00


In [67]:
import pandas as pd
import numpy as np



import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV

from sklift.models import SoloModel, TwoModels, ClassTransformation
from sklift.metrics import qini_auc_score
#from sklift.viz import plot_qini_curve

In [4]:
import os

os.listdir('/content')

['.config', 'ДЗ по Uplift обновленное', 'Data.zip', 'sample_data']

In [7]:
# ваш код здесь

DATA_PATH = "/content/ДЗ по Uplift обновленное"

aggs = pd.read_csv(f"{DATA_PATH}/AGGS_FINAL.csv")
campaigns = pd.read_csv(f"{DATA_PATH}/CAMPAINGS.csv")
contracts = pd.read_csv(f"{DATA_PATH}/CONTRACTS_FINAL.csv")
people = pd.read_csv(f"{DATA_PATH}/PEOPLE_IN_CAMPAINGS_FINAL.csv")

In [9]:
for df in [aggs, campaigns, contracts, people]:
    if "Unnamed: 0" in df.columns:
        df.drop(columns=["Unnamed: 0"], inplace=True)

aggs["report_dt"] = pd.to_datetime(aggs["report_dt"])
contracts["contract_date"] = pd.to_datetime(contracts["contract_date"])

people["delivery_date"] = pd.to_datetime(people["delivery_date"], errors="coerce")

campaigns["product_id"] = campaigns["product_id"].astype(int)
contracts["product_id"] = contracts["product_id"].astype(int)

In [10]:
campaigns_1 = campaigns[campaigns["product_id"] == 1].copy()

campaigns_1

,campaing_id,product_id,channel
0,iddqd,1,push
1,idclip,1,sms
2,iddt,1,banner
3,idkfa,1,other_ads


In [12]:
people_campaigns = people.merge(campaigns_1, on="campaing_id", how="inner")

people_campaigns

,campaing_id,user_id,t_flag,delivery_date,product_id,channel
0,idclip,1099975,1,2024-11-06,1,sms
1,iddqd,1162,1,2024-11-08,1,push
2,iddqd,42991,1,2024-11-07,1,push
3,idclip,142343,0,NaT,1,sms
4,iddqd,24623,0,NaT,1,push
...,...,...,...,...,...,...
519995,iddt,4108275,1,2024-11-04,1,banner
519996,iddqd,131927,1,2024-11-06,1,push
519997,idclip,1074765,1,2024-11-05,1,sms
519998,iddqd,73995,0,NaT,1,push


In [13]:
delivery_check = (
    people_campaigns.groupby(["channel", "t_flag"]).agg(
        rows=("user_id", "count"),
        missing_delivery=("delivery_date", lambda x: x.isna().sum()),
        min_delivery=("delivery_date", "min"),
        max_delivery=("delivery_date", "max")
    ).reset_index())

delivery_check

,channel,t_flag,rows,missing_delivery,min_delivery,max_delivery
0,banner,0,60000,60000,NaT,NaT
1,banner,1,60000,0,2024-11-04,2024-11-08
2,other_ads,0,60000,60000,NaT,NaT
3,other_ads,1,60000,0,2024-11-04,2024-11-08
4,push,0,80000,80000,NaT,NaT
5,push,1,80000,0,2024-11-04,2024-11-08
6,sms,0,60000,60000,NaT,NaT
7,sms,1,60000,0,2024-11-04,2024-11-08


In [14]:
people_campaigns.groupby(["channel", "t_flag"]).size()

channel    t_flag
banner     0         60000
           1         60000
other_ads  0         60000
           1         60000
push       0         80000
           1         80000
sms        0         60000
           1         60000
dtype: int64

In [15]:
contracts_1 = contracts[contracts["product_id"] == 1].copy()

contracts_1.head()

,user_id,contract_date,product_id,contract_id
0,4008279,2024-11-03,1,0001_2024-11-03_4008279
1,2079035,2024-11-08,1,0001_2024-11-08_2079035
2,103088,2024-11-13,1,0001_2024-11-13_103088
3,2026788,2024-11-02,1,0001_2024-11-02_2026788
4,52269,2024-11-17,1,0001_2024-11-17_52269


In [16]:
base = people_campaigns.merge(
    contracts_1[["user_id", "contract_id", "contract_date"]],
    on="user_id",
    how="left"
)

base.head()

,campaing_id,user_id,t_flag,delivery_date,product_id,channel,contract_id,contract_date
0,idclip,1099975,1,2024-11-06,1,sms,0001_2024-11-11_1099975,2024-11-11
1,iddqd,1162,1,2024-11-08,1,push,0001_2024-11-13_1162,2024-11-13
2,iddqd,42991,1,2024-11-07,1,push,NaN,NaT
3,idclip,142343,0,NaT,1,sms,0001_2024-11-17_142343,2024-11-17
4,iddqd,24623,0,NaT,1,push,NaN,NaT


In [17]:
campaign_start = pd.Timestamp("2024-11-01")
feature_report_dt = pd.Timestamp("2024-09-30")

base["target_purchase"] = 0

mask_treatment = ((base["t_flag"] == 1) & base["delivery_date"].notna() & base["contract_date"].notna() & (base["contract_date"] >= base["delivery_date"]) & (base["contract_date"] <= base["delivery_date"] + pd.Timedelta(days=14)))

mask_control = ((base["t_flag"] == 0) & base["contract_date"].notna() & (base["contract_date"] >= campaign_start) & (base["contract_date"] <= campaign_start + pd.Timedelta(days=21)))

base.loc[mask_treatment | mask_control, "target_purchase"] = 1

In [18]:
target_table = (base.groupby(["user_id", "campaing_id", "channel", "t_flag"], as_index=False).agg(target=("target_purchase", "max")))

target_table["report_dt"] = feature_report_dt

target_table.head(), target_table.shape

(   user_id campaing_id channel  t_flag  target  report_dt
 0        1      idclip     sms       0       0 2024-09-30
 1        1       iddqd    push       1       0 2024-09-30
 2        2       iddqd    push       0       0 2024-09-30
 3        3       iddqd    push       0       0 2024-09-30
 4        4       iddqd    push       0       0 2024-09-30,
 (520000, 6))

In [19]:
target_by_channel = target_table.rename(columns={"user_id": "client_id"})[
    ["client_id", "report_dt", "channel", "t_flag", "target"]
]

group_stats = (
    target_by_channel.groupby(["channel", "t_flag"]).agg(
        clients=("client_id", "nunique"),
        target_mean=("target", "mean"),
        target_sum=("target", "sum")
    ).reset_index()
)

group_stats

,channel,t_flag,clients,target_mean,target_sum
0,banner,0,60000,0.400733,24044
1,banner,1,60000,0.602717,36163
2,other_ads,0,60000,0.400733,24044
3,other_ads,1,60000,0.602717,36163
4,push,0,80000,0.202150,16172
5,push,1,80000,0.601738,48139
6,sms,0,60000,0.684917,41095
7,sms,1,60000,0.201167,12070


In [20]:
uplift_table = (target_by_channel.groupby(["channel", "t_flag"])["target"].mean().unstack().rename(columns={0: "control_rate", 1: "treatment_rate"}))

uplift_table["uplift"] = (uplift_table["treatment_rate"] - uplift_table["control_rate"])

sizes = (target_by_channel.groupby(["channel", "t_flag"])["client_id"].nunique().unstack().rename(columns={0: "control_size", 1: "treatment_size"}))
uplift_summary = uplift_table.merge(sizes,left_index=True, right_index=True, how="left")

uplift_summary = uplift_summary.sort_values("uplift", ascending=False)

uplift_summary

t_flag,control_rate,treatment_rate,uplift,control_size,treatment_size
channel,,,,,
push,0.202150,0.601738,0.399588,80000,80000
banner,0.400733,0.602717,0.201983,60000,60000
other_ads,0.400733,0.602717,0.201983,60000,60000
sms,0.684917,0.201167,-0.483750,60000,60000


In [21]:
best_channel = uplift_summary.index[0]
best_uplift = uplift_summary.loc[best_channel, "uplift"]

print(f"bets channel: {best_channel}")
print(f"Uplift: {best_uplift:.4f}")

bets channel: push
Uplift: 0.3996


In [22]:
campaign_rates = (
    target_table
    .groupby(["campaing_id", "channel", "t_flag"])
    .agg(
        n_clients=("user_id", "nunique"),
        target_rate=("target", "mean"),
        n_positive=("target", "sum")
    )
    .reset_index()
)

campaign_rates

,campaing_id,channel,t_flag,n_clients,target_rate,n_positive
0,idclip,sms,0,60000,0.684917,41095
1,idclip,sms,1,60000,0.201167,12070
2,iddqd,push,0,80000,0.202150,16172
3,iddqd,push,1,80000,0.601738,48139
4,iddt,banner,0,60000,0.400733,24044
5,iddt,banner,1,60000,0.602717,36163
6,idkfa,other_ads,0,60000,0.400733,24044
7,idkfa,other_ads,1,60000,0.602717,36163


In [27]:
user_channel_counts = (target_by_channel.groupby("client_id")["channel"].nunique().reset_index(name="n_channels"))

user_channel_counts["n_channels"].value_counts().sort_index()

,count
n_channels,
1,400000
2,60000


In [24]:
multi_channel_users = user_channel_counts[
    user_channel_counts["n_channels"] > 1
]

multi_channel_users.shape

(60000, 2)

In [28]:
multi_channel_ids = multi_channel_users["client_id"]

target_by_channel[target_by_channel["client_id"].isin(multi_channel_ids)].groupby(["channel", "t_flag"]).size()

,,0
channel,t_flag,
push,1,60000
sms,0,60000


In [29]:
target_by_channel[target_by_channel["client_id"].isin(multi_channel_ids)].sort_values(["client_id", "channel"]).head(20)

,client_id,report_dt,channel,t_flag,target
1,1,2024-09-30,push,1,0
0,1,2024-09-30,sms,0,0
6,5,2024-09-30,push,1,0
5,5,2024-09-30,sms,0,0
8,6,2024-09-30,push,1,1
7,6,2024-09-30,sms,0,1
13,10,2024-09-30,push,1,1
12,10,2024-09-30,sms,0,1
18,14,2024-09-30,push,1,1
17,14,2024-09-30,sms,0,1


In [30]:
broken_channels = ["sms"]

valid_target = target_by_channel[~target_by_channel["channel"].isin(broken_channels)].copy()

valid_target["channel"].value_counts()

,count
channel,
push,160000
other_ads,120000
banner,120000


### ваши выводы здесь

максимальный uplift показал канал push: 0.3996(конверсия в контрольной группе составила 0.2022 а в целевой группе 0.6017)

каналы banner и other_ads показали одинаковый uplift 0.2020, их агрегированные статистики полностью совпадают, но явного пересечения клиентов между нимим я обнаружить не смог

канал sms показал отрицательный uplift -0.4838, но при просмотре пересечений было обнаружено что 60000 людей одновременно находятся в целевой группе push и в контрольной группе sms $\Longrightarrow$ контрольная группа sms загрязнена push-jv. значит оценка эффекта sms некорректна, и отрицательный uplift нельзя напрямую интерпретировать как вредн

<h2> 2. Клиентские агрегаты (12 баллов)</h2>

Присоедините клиентские агрегаты (будьте внимательны, присоедините агрегаты за корректный месяц) и изучите полученные данные.

**Комментарий по заданиям и оцениванию:**

* Вы должны корректно присоединить клиентские агрегаты со смещением на два месяца, чтобы не было лика таргета. За данное действие можно получить **4 балла**

* Далее вы должен сделать UPLIFT EDA, которые обсуждались на лекции и показывались в практических ноутбуках. В ходе анализа вы должны проверить корректность данных по рекламным кампаниям и решить, что делать со "сломанными" кампаниями. По итогам анализа подготовьте выводы. За данное действие можно получить **8 баллов**

In [47]:
# ваш код здесь

target_table["report_dt"] = pd.Timestamp("2024-09-30")
aggs["report_dt"] = pd.to_datetime(aggs["report_dt"])

data_full = target_table.merge(aggs, on=["user_id", "report_dt"], how="left")

print("target_table:", target_table.shape)
print("data_full:", data_full.shape)

target_table: (520000, 6)
data_full: (520000, 17)


In [48]:
feature_cols = ["x1", "x2", "x3", "x4", "x5", "x6", "x7", "x8", "x9", "age", "city"]

data_full[feature_cols].isna().mean()

,0
x1,0.0
x2,0.0
x3,0.0
x4,0.0
x5,0.0
x6,0.0
x7,0.0
x8,0.0
x9,0.0
age,0.0


In [54]:
num_features = ["x1", "x2", "x3", "x4", "x5", "x6", "x7", "x8", "x9", "age"]

balance_rows = []

for channel in data_full["channel"].unique():
    df_ch = data_full[data_full["channel"] == channel]

    for col in num_features:
        control = df_ch[df_ch["t_flag"] == 0][col]
        treatment = df_ch[df_ch["t_flag"] == 1][col]

        control_mean = control.mean()
        treatment_mean = treatment.mean()
        control_std = control.std()
        treatment_std = treatment.std()

        pooled_std = np.sqrt((control_std ** 2 + treatment_std ** 2) / 2)

        smd = (treatment_mean - control_mean) / pooled_std if pooled_std != 0 else np.nan

        balance_rows.append({
            "channel": channel,
            "feature": col,
            "control_mean": control_mean,
            "treatment_mean": treatment_mean,
            "diff": treatment_mean - control_mean,
            "smd": smd,
            "abs_smd": abs(smd)
        })

balance_df = pd.DataFrame(balance_rows)

balance_df.sort_values("abs_smd", ascending=False).head(30)

,channel,feature,control_mean,treatment_mean,diff,smd,abs_smd
9,sms,age,32.459500,32.538800,0.079300,0.010590,0.010590
11,push,x2,0.991262,1.004994,0.013732,0.010553,0.010553
29,other_ads,age,32.565050,32.494200,-0.070850,-0.009451,0.009451
34,banner,x5,0.005851,-0.003222,-0.009073,-0.009099,0.009099
19,push,age,32.504013,32.448462,-0.055550,-0.007413,0.007413
30,banner,x1,-0.200788,-0.190153,0.010635,0.006692,0.006692
20,other_ads,x1,-0.200788,-0.190153,0.010635,0.006692,0.006692
21,other_ads,x2,-1.005438,-0.996064,0.009374,0.006497,0.006497
31,banner,x2,-1.005438,-0.996064,0.009374,0.006497,0.006497
13,push,x4,-0.606210,-0.596487,0.009724,0.006202,0.006202


In [55]:
balance_summary = (
    balance_df
    .groupby("channel")
    .agg(
        max_abs_smd=("abs_smd", "max"),
        mean_abs_smd=("abs_smd", "mean"),
        bad_features=("abs_smd", lambda x: (x > 0.1).sum())
    )
    .reset_index()
    .sort_values("max_abs_smd", ascending=False)
)

balance_summary

,channel,max_abs_smd,mean_abs_smd,bad_features
3,sms,0.010590,0.001716,0
2,push,0.010553,0.004262,0
1,other_ads,0.009451,0.003969,0
0,banner,0.009099,0.003889,0


In [58]:
data_full["age_bin"] = pd.cut(
    data_full["age"],
    bins=[0, 25, 35, 45, 55, 65, 100],
    labels=["<=25", "26-35", "36-45", "46-55", "56-65", "65+"]
)

age_uplift = (
    data_full.groupby(["channel", "age_bin", "t_flag"], observed=False)["target"].mean().unstack().rename(columns={0: "control_rate", 1: "treatment_rate"})
)

age_uplift["uplift"] = age_uplift["treatment_rate"] - age_uplift["control_rate"]

age_uplift.reset_index().sort_values(["channel", "age_bin"])

t_flag,channel,age_bin,control_rate,treatment_rate,uplift
0,banner,<=25,0.401230,0.612929,0.211699
1,banner,26-35,0.401483,0.598433,0.196950
2,banner,36-45,0.399689,0.600860,0.201171
3,banner,46-55,NaN,NaN,NaN
4,banner,56-65,NaN,NaN,NaN
5,banner,65+,NaN,NaN,NaN
6,other_ads,<=25,0.398788,0.606006,0.207218
7,other_ads,26-35,0.403746,0.603962,0.200216
8,other_ads,36-45,0.398912,0.599498,0.200585
9,other_ads,46-55,NaN,NaN,NaN


In [59]:
city_uplift = (
    data_full
    .groupby(["channel", "city", "t_flag"])["target"]
    .mean()
    .unstack()
    .rename(columns={0: "control_rate", 1: "treatment_rate"})
)

city_uplift["uplift"] = city_uplift["treatment_rate"] - city_uplift["control_rate"]

city_sizes = (
    data_full
    .groupby(["channel", "city", "t_flag"])["user_id"]
    .nunique()
    .unstack()
    .rename(columns={0: "control_size", 1: "treatment_size"})
)

city_uplift = city_uplift.merge(
    city_sizes,
    left_index=True,
    right_index=True,how="left"
).reset_index()

city_uplift_filtered = city_uplift[(city_uplift["control_size"] >= 1000) & (city_uplift["treatment_size"] >= 1000)]

city_uplift_filtered.sort_values("uplift", ascending=False).head(30)

t_flag,channel,city,control_rate,treatment_rate,uplift,control_size,treatment_size
8,push,Ufa,0.200632,0.601412,0.400780,26915,26772
7,push,Smolensk,0.200834,0.600677,0.399843,26380,26585
6,push,Moscow,0.204980,0.603123,0.398142,26705,26643
1,banner,Smolensk,0.399930,0.608383,0.208453,20036,20017
4,other_ads,Smolensk,0.396772,0.601370,0.204598,36056,20006
0,banner,Moscow,0.399118,0.601773,0.202655,19954,19858
5,other_ads,Ufa,0.408866,0.606196,0.197330,11911,20175
3,other_ads,Moscow,0.404554,0.600535,0.195981,12033,19819
2,banner,Ufa,0.403148,0.598012,0.194864,20010,20125
11,sms,Ufa,0.680198,0.204682,-0.475516,19978,20075


### ваши выводы здесь

после объединения к-во строк не изменилось а доля пропусков в признаках x1–x9, age и city составила 0.0. те для всех объектов были найдены клиентские агрегаты за нужнвйы месяц

**DISCLAIMER**: к сожалению я в презентациах не смог найти/ не увидел uplift eda пришлось гуглить очень быстро, очнь вероятно я неправильно вообще понял в чем его смысл

Для проверки корректности разбиения клиентов на treatment и control был рассчитан standardized mean difference по x1–x9 и age, во всех каналах макс значение abs(SMD) оказалось меньше 0.011, а к-во признаков с abs(SMD) > 0.1 равно нулю. знаичт treatment и control группы хорошо сбалансированы поагрегатим

был рассчитан uplift в разрезе возрастных групп. push показал стабильно высокий положительный uplift во всех доступных возрастных группах: примерно от 0.393 до 0.404(Banner и other_ads дают положительный uplift. SMS имеет отрицательный uplift во всех городах.)

uplift по городам показал аналогичную картину. Push является наиболее эффективным каналом в Москве, Уфе и Смоленске: uplift находится около 0.40(Banner и other_ads дают положительный uplift. SMS имеет отрицательный uplift во всех городах.)

<h2> 3. Построение моделей и оценка их качества (14 баллов)</h2>

Постройте Uplift модели по собранным кампаниям, проведите тюнинг гиперпараметров и оцените их качество (qini score). Для каждой модели также постройте qini-curve.

**Комментарий по заданиям и оцениванию:**

* Реализован только подход Solomodel без дополнительных библиотек и калибровок  - **1 балл**

* Реализован Solomodel или Twomodel через Sklift или CausalML - **2 балла**

* Учтена калибровка Metalearner'ах - **2 балла**

* Корректно реализован ClassTransformation - **2 балла**

* Реализован UpliftRandomForest - **4 балла**

* Использованы пайплайны в Sklift - **2 балла**

* Реализован тюнинг ( Gridsearch \ Optuna ) - **1 балл**

In [73]:
# ваш код здесь

def train_eval_uplift_model(df, channel, model_name, model):
    df_ch = df[df["channel"] == channel].copy()

    X = df_ch[feature_cols]
    y = df_ch["target"].astype(int)
    treatment = df_ch["t_flag"].astype(int)

    X_train, X_test, y_train, y_test, tr_train, tr_test = train_test_split(X, y, treatment, test_size=0.3, random_state=42, stratify=treatment)

    model.fit(X_train, y_train, tr_train)

    uplift_pred = model.predict(X_test)

    qini = qini_auc_score(
        y_true=y_test,
        uplift=uplift_pred,
        treatment=tr_test
    )

    print(f"Channel: {channel}")
    print(f"Model: {model_name}")
    print(f"Qini AUC: {qini:.5f}")

    return {
        "channel": channel,
        "model": model_name,
        "qini_auc": qini,
        "fitted_model": model
    }



я не успеваю :(

<h2>4. Подготовка ответа в требуемом формате и подготовка выводов (6 баллов)</h2>

a) Сделайте скоринг нужных клиентов, подготовьте ответ в требуемом формате

б) Сделайте краткую аналитику того, какой канал взаимодействия наиболее предпочтителен

в) Сделайте выводы по проделанной работе

**Комментарий по заданиям и оцениванию:**

* Подготовлен только ответ - **1 балл**
* Подготовлен содержательный вывод по проделанной работе - **4 балла**
* Корректно принято решение об отправке/не отправке коммуникации клиентам в зависимости от значений Uplift - **1 балл**

In [ ]:
# ваш код здесь

### ваши выводы здесь